# Microwave dielectric properties of natural materials

An undergraduate computational companion to Ulaby & Long, Chapter 4. The notebook moves from pure substances to mixtures, then applies the models to ice, snow, rock, vegetation, and canopies. NavaSAR writes passive loss as $\epsilon=\epsilon'+j\epsilon''$. Some references use the opposite sign because they assume a different time convention.

In [ ]:
import sys
from pathlib import Path
try:
    import navasar
except ModuleNotFoundError:
    root = next((p for p in (Path.cwd(), Path.cwd().parent) if (p/'navasar').is_dir()), None)
    if root is None:
        raise ModuleNotFoundError('Run: python -m pip install -e .[notebooks]')
    sys.path.insert(0, str(root))
import numpy as np
import matplotlib.pyplot as plt
from navasar.materials import *
plt.rcParams.update({'figure.figsize': (8, 4.5), 'axes.grid': True})
print('Python:', sys.executable)
print('NavaSAR:', navasar.__file__)

## 1. Storage and loss

The real part $\epsilon'$ governs phase velocity and refraction. The loss factor $\epsilon''$ controls absorption. Liquid-water molecules rotate in response to an alternating field, producing a strong frequency-dependent relaxation. Ice cannot reorient as freely and is much less lossy.

In [ ]:
f = np.logspace(-1, 2, 300)
water = pure_water_permittivity(f, 20)
ice = pure_ice_permittivity(f, -10)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].semilogx(f, water.real, label='water, 20°C'); ax[0].semilogx(f, ice.real, label='ice, -10°C')
ax[1].loglog(f, water.imag, label='water, 20°C'); ax[1].loglog(f, ice.imag, label='ice, -10°C')
ax[0].set(xlabel='Frequency (GHz)', ylabel="Permittivity ε'"); ax[1].set(xlabel='Frequency (GHz)', ylabel="Loss factor ε''")
for a in ax: a.legend(); a.grid(True, which='both', alpha=.3)
fig.suptitle('Liquid water and pure ice respond very differently')

## 2. Temperature changes water and ice differently

Water relaxation shifts with temperature. Pure-ice permittivity stays near 3.2, but its already-small loss changes substantially. Always keep frequency and temperature attached to a dielectric value.

In [ ]:
freq = np.logspace(-1, 1.5, 200)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for T in (0, 10, 20, 30):
    eps = pure_water_permittivity(freq, T); ax[0].semilogx(freq, eps.real, label=f'{T}°C')
for T in (-1, -10, -20, -30):
    eps = pure_ice_permittivity(freq, T); ax[1].loglog(freq, eps.imag, label=f'{T}°C')
ax[0].set(xlabel='Frequency (GHz)', ylabel="Water ε'"); ax[1].set(xlabel='Frequency (GHz)', ylabel="Ice ε''")
for a in ax: a.legend(); a.grid(True, which='both', alpha=.3)

## 3. Inclusion geometry

A mixture is not determined by its ingredients alone. An ellipsoid has depolarization factors $L_a+L_b+L_c=1$. Spheres respond equally along every axis; discs and needles do not. Random orientation averages the three responses.

In [ ]:
fraction = np.linspace(0, .35, 120)
shapes = {'disc (1,1,0.1)': (1,1,.1), 'sphere (1,1,1)': (1,1,1), 'needle (1,1,8)': (1,1,8)}
for label, axes in shapes.items():
    L = depolarization_factors(*axes)
    mixed = dilute_ellipsoid_mixing(2.5+.02j, 15+2j, fraction, L)
    plt.plot(fraction, mixed.real, label=f'{label}; L={np.round(L,2)}')
plt.xlabel('Inclusion volume fraction'); plt.ylabel("Effective ε'"); plt.legend(); plt.grid(alpha=.3)

## 4. Mixing rules are assumptions

The dilute model neglects strong inclusion interactions. Polder–van Santen is symmetric between phases. TVB imposes a host/inclusion geometry. Their disagreement is a useful estimate of structural-model uncertainty, not a numerical error.

In [ ]:
vf = np.linspace(0, 1, 151)
host, inclusion = 1+0j, 10+1j
dilute = dilute_ellipsoid_mixing(host, inclusion, vf)
pvs = np.array([polder_van_santen([host, inclusion], [1-x, x]) for x in vf])
tvb = tvb_mixing(host, inclusion, vf, 'spheres')
refr = np.array([refractive_mixture([host, inclusion], [1-x, x]) for x in vf])
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for y, name in ((dilute,'dilute'),(pvs,'Polder–van Santen'),(tvb,'TVB spheres'),(refr,'refractive')):
    ax[0].plot(vf, y.real, label=name); ax[1].plot(vf, y.imag, label=name)
ax[0].set(xlabel='Inclusion fraction', ylabel="ε'"); ax[1].set(xlabel='Inclusion fraction', ylabel="ε''")
for a in ax: a.legend(); a.grid(alpha=.3)

## 5. Sea ice: temperature controls brine

Freezing excludes salt from the ice lattice and concentrates it in liquid pockets. Cooling changes equilibrium brine salinity and brine volume. Even a small liquid fraction can dominate dielectric loss. This compact model is not a substitute for first-year/multiyear ice structure.

In [ ]:
temperature = np.linspace(-22.8, -2, 180)
salinity = brine_salinity(temperature)
volume = brine_volume_fraction(5.0, temperature)
eps_si = np.array([sea_ice_permittivity(5.4, T, 5.0) for T in temperature])
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].plot(temperature, salinity); ax[0].set(ylabel='Brine salinity (psu)')
ax[1].plot(temperature, 100*volume); ax[1].set(ylabel='Brine volume (%)')
ax[2].semilogy(temperature, eps_si.imag); ax[2].set(ylabel="Sea-ice ε'' at 5.4 GHz")
for a in ax: a.set_xlabel('Temperature (°C)'); a.grid(alpha=.3)

## 6. Dry and wet snow

Dry snow is mostly air and ice, so density controls its real permittivity. Adding only a few percent liquid water causes a large loss increase. Radar sensitivity therefore changes sharply during melt.

In [ ]:
rho = np.linspace(.05, .7, 160)
empirical = dry_snow_permittivity(rho)
tvb_snow = dry_snow_tvb(9.6, rho, -10)
wetness = np.linspace(0, .12, 80)
wet = np.array([wet_snow_permittivity(9.6, .25, w) for w in wetness])
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(rho, empirical.real, label='empirical density law'); ax[0].plot(rho, tvb_snow.real, '--', label='TVB ice/air')
ax[1].semilogy(100*wetness, wet.imag)
ax[0].set(xlabel='Dry density (g cm⁻³)', ylabel="Dry-snow ε'"); ax[1].set(xlabel='Liquid water by volume (%)', ylabel="Wet-snow ε''")
ax[0].legend(); [a.grid(alpha=.3) for a in ax]

## 7. Solid rock versus powdered rock

Crushing rock replaces solid volume with air. Bulk density is therefore a proxy for porosity. This example separates the intrinsic solid permittivity from the effective permittivity measured for a powder.

In [ ]:
bulk_density = np.linspace(.2, 2.65, 160)
for eps_solid in (4, 6, 9):
    eps_powder = powdered_rock_permittivity(eps_solid, bulk_density)
    plt.plot(bulk_density, eps_powder.real, label=f'solid ε={eps_solid}')
plt.axvline(1, color='k', ls=':', label='typical powder density')
plt.xlabel('Bulk density (g cm⁻³)'); plt.ylabel("Powder effective ε'"); plt.legend(); plt.grid(alpha=.3)

## 8. Vegetation: free water, bound water, and air

Plant water is divided into free and molecularly bound components. The vegetation-material model first combines residual dry matter with both water states. A second mixing step dilutes plant material with air to represent a sparse canopy.

In [ ]:
moisture = np.linspace(0, .9, 160)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for fi in (1.4, 5.4, 10):
    plant = vegetation_permittivity(fi, moisture)
    ax[0].plot(moisture, plant.real, label=f'{fi} GHz')
    ax[1].plot(moisture, plant.imag, label=f'{fi} GHz')
ax[0].set(xlabel='Gravimetric moisture', ylabel="Plant material ε'"); ax[1].set(xlabel='Gravimetric moisture', ylabel="Plant material ε''")
for a in ax: a.legend(); a.grid(alpha=.3)
plant = vegetation_permittivity(5.4, .7)
for vf in (.01, .03, .08): print(f'Canopy fraction {vf:.0%}: ε = {canopy_effective_permittivity(plant, vf)}')

## 9. A small sensitivity experiment

A useful modeling habit is to vary uncertain physical inputs instead of reporting a single curve. Below, wet-snow loss is evaluated across density and water fraction. The result reveals where uncertainty in water content matters most.

In [ ]:
density_grid = np.linspace(.1, .5, 60)
water_grid = np.linspace(0, .1, 61)
loss = np.array([[wet_snow_permittivity(5.4, r, w).imag for w in water_grid] for r in density_grid])
plt.figure(figsize=(8,4.5)); image = plt.pcolormesh(100*water_grid, density_grid, loss, shading='auto', cmap='magma')
plt.xlabel('Liquid-water fraction (%)'); plt.ylabel('Dry density (g cm⁻³)'); plt.colorbar(image, label="ε'' at 5.4 GHz"); plt.grid(False)

## 10. What to remember

1. A dielectric value is incomplete without frequency, temperature, composition, and sign convention.
2. Inclusion geometry can matter as much as volume fraction.
3. Water—especially saline or liquid water—often dominates microwave loss.
4. Plant material and a whole canopy are different effective media.
5. Disagreement between defensible mixing rules is model uncertainty worth reporting.
6. Effective-medium models predict bulk propagation; they do not automatically predict backscatter, which also depends on interfaces, roughness, geometry, and spatial fluctuations.